# Feature Engineering
Create a Dataframe for model input:
- Feature creation
- Daily aggregation of features
- Vizualization

# Import Libraries and Data

In [13]:
# --- Standard Libraries ---
import os
import re
from collections import Counter

# --- Data Science ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# --- NLP ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect, DetectorFactory, LangDetectException

# --- Transformers ---
import torch
from torch.nn.functional import sigmoid
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertForSequenceClassification
)
from scipy.special import softmax

# --- Utils ---
from tqdm.auto import tqdm
from IPython.display import display

# --- Setup ---
tqdm.pandas()
nltk.download("punkt")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\chrii\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\chrii\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Kontrollvariablen

In [14]:
# Uses shorter time periods for testing purposes
TEST = False

# Wenn nur einzelne Spalten/Features hinzugefügt werden sollen, bitte unten den Block 'neue Features zur CSV hinzufügen' entsprechend anpassen. Die bestehende final_daily_df csv wird dann in ein df geladen und die neuen Spalten werden dazugemerged und die csv wieder abgespeichert.
einzelne_features_zur_bestehenden_CSV_hinzufügen = False # default = False

# wenn True, wird die final_daily_df CSV ganz neu zusammengestellt.
vollstaendige_neuerstellung_der_csv = True # default = False

In [15]:
musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["isRetweet"] = df["isRetweet"].astype(str).str.lower()
    df["possiblySensitive"] = df["possiblySensitive"].astype(str).str.lower()
    df["fullText"] = df["fullText"].astype(str)

musk_twitter_data_nlp["text_raw"] = musk_twitter_data_nlp["text_raw"].astype(str)
musk_twitter_data_nlp["text_lemmatized"] = musk_twitter_data_nlp["text_lemmatized"].astype(str)

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["date"] = df["createdAt"].dt.date

if TEST:
    start_date = pd.to_datetime("2025-04-01").date()
else:
    start_date = pd.to_datetime("2015-01-01").date()

end_date = musk_twitter_data_all["date"].max()

mask_all = (musk_twitter_data_all["date"] >= start_date) & (musk_twitter_data_all["date"] <= end_date)
musk_twitter_data_all = musk_twitter_data_all.loc[mask_all].reset_index(drop=True)

mask_nlp = (musk_twitter_data_nlp["date"] >= start_date) & (musk_twitter_data_nlp["date"] <= end_date)
musk_twitter_data_nlp = musk_twitter_data_nlp.loc[mask_nlp].reset_index(drop=True)

final_daily_df_base = pd.DataFrame({
    'date': pd.date_range(start=start_date, end=end_date)
})
final_daily_df_base["date"] = final_daily_df_base["date"].dt.date 

print("NLPTweets:", musk_twitter_data_nlp.shape, "AllTweets:", musk_twitter_data_all.shape)
musk_twitter_data_nlp.info()
musk_twitter_data_all.info()

C:\Users\chrii\AppData\Local\Temp\ipykernel_13176\1047191983.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
C:\Users\chrii\AppData\Local\Temp\ipykernel_13176\1047191983.py:2: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])


NLPTweets: (41823, 29) AllTweets: (54023, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41823 entries, 0 to 41822
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   id                        41823 non-null  int64              
 1   url                       41823 non-null  object             
 2   twitterUrl                41823 non-null  object             
 3   fullText                  41823 non-null  object             
 4   retweetCount              41772 non-null  float64            
 5   replyCount                41225 non-null  float64            
 6   likeCount                 41772 non-null  float64            
 7   quoteCount                41214 non-null  float64            
 8   viewCount                 25981 non-null  float64            
 9   createdAt                 41823 non-null  datetime64[ns, UTC]
 10  bookmarkCount             41214 non-

# Tweet activity
New features:
- Number of tweets per day

In [16]:
tweet_counts_daily = (
    musk_twitter_data_all
    .groupby("date")
    .size()
    .reset_index(name="tweet_count")
)

# Engagement metrics
- like_count
- quoted_count
- retweet_count
- view_count

In [17]:
# Engagement metrics calculation: Like, Quote, Retweet, Reply counts per day
engagement_metrics = (
    musk_twitter_data_all
    .groupby('date')[['likeCount', 'quoteCount', 'retweetCount', 'replyCount']]
    .sum()
    .astype(int)
    .reset_index()
)


# Sentiment Analysis

New Features:
- Poitve, Neutral ans Negative percentage of posts
- Polarization: Tweets with pos/neg > 0,6

In [18]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm

# Setup
tqdm.pandas()

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Load model/tokenizer and move model to device
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()  # deactivate dropout etc.

# Preprocessing helper
def preprocess(text):
    return text.replace("\n", " ").strip()

# Sentiment inference
def get_sentiment_probs(text):
    text = preprocess(text)
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.detach().cpu().numpy()[0])
    return {
        "sentiment": ['negative', 'neutral', 'positive'][probs.argmax()],
        "neg": probs[0],
        "neu": probs[1],
        "pos": probs[2],
    }

# Optional: label for strong polarity
def polarized_label(row):
    return "polarized" if max(row["pos"], row["neg"]) > 0.6 else "not_polarized"

# --- Analysis Start ---

# Sentiment scoring
results = musk_twitter_data_nlp["text_raw"].progress_apply(get_sentiment_probs).apply(pd.Series)
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp, results], axis=1)

# Polarity classification
max_sent = musk_twitter_data_nlp[["pos", "neg"]].max(axis=1)
musk_twitter_data_nlp["sentiment_polarity"] = np.where(max_sent > 0.6, "polarized", "not_polarized")

# 1) Unweighted aggregation
sentiment_avg = musk_twitter_data_nlp.groupby("date")[["neg", "neu", "pos"]].mean().reset_index()
nlp_counts = musk_twitter_data_nlp.groupby("date").size().reset_index(name="nlp_tweet_count")
polar_mean = (
    musk_twitter_data_nlp.groupby("date")["sentiment_polarity"]
    .apply(lambda s: (s == "polarized").mean())
    .reset_index(name="polarized")
)

sentiment_daily = (
    sentiment_avg
    .merge(nlp_counts, on="date", how="left")
    .merge(polar_mean, on="date", how="left")
)

# 2) Weighted aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(
        neg_w=lambda df: df["neg"] * df["engagement_index"],
        neu_w=lambda df: df["neu"] * df["engagement_index"],
        pos_w=lambda df: df["pos"] * df["engagement_index"]
    )
    .groupby("date")
    .agg(
        neg_w_sum=("neg_w", "sum"),
        neu_w_sum=("neu_w", "sum"),
        pos_w_sum=("pos_w", "sum"),
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
    .assign(
        neg=lambda df: df["neg_w_sum"] / df["total_engagement"],
        neu=lambda df: df["neu_w_sum"] / df["total_engagement"],
        pos=lambda df: df["pos_w_sum"] / df["total_engagement"],
    )
    .drop(columns=["neg_w_sum", "neu_w_sum", "pos_w_sum"])
)

# 2b) Weighted polarization
polar_weighted = (
    musk_twitter_data_nlp
    .groupby(["date", "sentiment_polarity"])["engagement_index"]
    .sum()
    .reset_index(name="eng_w_sum")
    .pivot(index="date", columns="sentiment_polarity", values="eng_w_sum")
    .fillna(0)
    .reset_index()
    .merge(weighted_sums[["date", "total_engagement"]], on="date", how="left")
    .assign(polarized=lambda df: df["polarized"] / df["total_engagement"])
    [["date", "polarized"]]
)

# 2c) Final weighted DataFrame
sentiment_daily_weighted = (
    weighted_sums[["date", "neg", "neu", "pos"]]
    .merge(polar_weighted, on="date", how="left")
    .merge(nlp_counts, on="date", how="left")
)


cuda


100%|██████████| 41823/41823 [09:59<00:00, 69.76it/s]


# Emotions & Personality

New Features:
- Ekman Emotions: anger, disgust, fear, joy, neutral, sadness, surprise
- Big 5 personality traits: Extroversion, Neuroticism, Agreeableness, Conscientiousness, Openness

In [19]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm

tqdm.pandas()

# Detect device (GPU oder CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model und Tokenizer laden
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# Emotionen extrahieren (Einzeltweet → Wahrscheinlichkeiten)
def get_emotions(text):
    text = str(text).replace("\n", " ").strip()
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**tokens).logits
    probs = softmax(logits.detach().cpu().numpy()[0])
    return dict(zip(emotion_labels, probs))

# Emotionen berechnen
print("Calculating emotion probabilities...")
emotion_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_emotions).apply(pd.Series)

# DataFrame erweitern
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), emotion_probs], axis=1)

# Ungewichtete Tagesaggregation
print("Aggregating daily emotions...")
emotion_daily = (
    musk_twitter_data_nlp
    .groupby('date')[emotion_labels]
    .mean()
    .reset_index()
)

# Gewichtete Tagesaggregation
print("Aggregating daily emotions (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{emo}_w": musk_twitter_data_nlp[emo] * musk_twitter_data_nlp["engagement_index"]
        for emo in emotion_labels
    })
    .groupby("date")
    .agg(
        **{f"{emo}_w_sum": (f"{emo}_w", "sum") for emo in emotion_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

# Normierung auf Engagement
emotion_daily_weighted = (
    weighted_sums
    .assign(**{
        emo: weighted_sums[f"{emo}_w_sum"] / weighted_sums["total_engagement"]
        for emo in emotion_labels
    })
    [["date", *emotion_labels]]
)

print("Done!")


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

C:\Users\chrii\PycharmProjects\COIN2025_VollVertnuetelt\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chrii\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Calculating emotion probabilities...


  1%|          | 348/41823 [00:03<05:57, 115.90it/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

100%|██████████| 41823/41823 [06:08<00:00, 113.41it/s]


Aggregating daily emotions...
Aggregating daily emotions (weighted)...
Done!


In [20]:
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
from scipy.special import expit as sigmoid  # sigmoid = 1 / (1 + exp(-x))
from tqdm import tqdm

tqdm.pandas()

# Detect device (GPU bevorzugt)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model und Tokenizer laden und auf das passende Gerät verschieben
model_name = "Minej/bert-base-personality"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

personality_labels = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']

# Funktion zur Extraktion der Big Five Scores
def get_personality(text):
    text = str(text).replace("\n", " ").strip()
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = sigmoid(outputs.logits.detach().cpu().numpy()).squeeze()
    return dict(zip(personality_labels, probs))

# Anwendung auf alle Texte
print("Calculating personality traits...")
personality_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_personality).apply(pd.Series)

# Daten anhängen
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), personality_probs], axis=1)

# Ungewichtete Aggregation
print("Aggregating daily personality...")
personality_daily = (
    musk_twitter_data_nlp
    .groupby('date')[personality_labels]
    .mean()
    .reset_index()
)

# Gewichtete Aggregation
print("Aggregating daily personality (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{pers}_w": musk_twitter_data_nlp[pers] * musk_twitter_data_nlp["engagement_index"]
        for pers in personality_labels
    })
    .groupby("date")
    .agg(
        **{f"{pers}_w_sum": (f"{pers}_w", "sum") for pers in personality_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

personality_daily_weighted = (
    weighted_sums
    .assign(**{
        pers: weighted_sums[f"{pers}_w_sum"] / weighted_sums["total_engagement"]
        for pers in personality_labels
    })
    [["date", *personality_labels]]
)

print("Done!")


tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

C:\Users\chrii\PycharmProjects\COIN2025_VollVertnuetelt\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chrii\.cache\huggingface\hub\models--Minej--bert-base-personality. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Calculating personality traits...


100%|██████████| 41823/41823 [10:06<00:00, 68.91it/s]


Aggregating daily personality...
Aggregating daily personality (weighted)...
Done!


# Topic and word counts

New Features: 
- Daily Word counts
    - Rationale of Definition of words:
        - Company/ticker terms (e.g. tesla, tsla, spacex) capture direct references to publicly traded entities.
        - Product names (e.g. model, cybertruck, starship) often precede news that can move stock prices.
        - Crypto tokens (e.g. bitcoin, dogecoin, ethereum, crypto) map to Musk-driven volatility in the digital-asset markets
        - Financial keywords (e.g. stock, market, price, profit, loss, revenue) directly signal earnings or valuation discussions.
        - Macro terms (e.g. inflation, interest) reflect broader economic commentary that can sway sentiment.
        - Action verbs (buy, sell) often presage trading intent or recommendations.
- Topics of posts

In [21]:
# Words
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|@\S+|[^a-z\s]", "", text)
    return text.split()

all_tokens = musk_twitter_data_nlp["text_lemmatized"].dropna().apply(tokenize)
flat_tokens = [token for sublist in all_tokens for token in sublist]
word_counts = Counter(flat_tokens)
word_counts = (
    pd.DataFrame(word_counts.items(), columns=["word", "count"])
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

top20 = [
    'tesla', 'stock', 'market', 'price', 'profit', 'loss', 'revenue',
    'inflation', 'interest', 'bitcoin', 'dogecoin', 'crypto', 'ethereum',
    'spacex', 'model', 'cybertruck', 'starship', 'buy', 'sell'
]

top_word_df = musk_twitter_data_nlp.dropna(subset=['text_lemmatized']).copy()
top_word_df['tokens'] = top_word_df['text_lemmatized'].apply(tokenize)
top_word_df = top_word_df.explode('tokens')
top_word_df['tokens'] = top_word_df['tokens'].replace({'tsla': 'tesla'})

top_word_df = top_word_df[top_word_df['tokens'].isin(top20)].copy()

daily_word_counts = (
    top_word_df
    .groupby(['date','tokens'])
    .size()
    .unstack(fill_value=0)
)

daily_word_counts = daily_word_counts.reindex(
    columns=top20,
    fill_value=0
).sort_index()

In [22]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm

tqdm.pandas()

# Device-Setup (GPU bevorzugt)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modell und Tokenizer laden
model_name = "cardiffnlp/tweet-topic-21-multi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

topic_labels = [
    "arts_culture", "business_entrepreneurs", "celebrity_pop_culture",
    "diaries_daily_life", "family", "fashion_style", "film_tv_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_educational",
    "music", "news_social_concern", "other_hobbies", "relationships",
    "science_technology", "sports", "travel_adventure", "youth_student_life"
]

# Klassifikationsfunktion
def get_topics(text):
    text = str(text).replace("\n", " ").strip()
    tokens = tokenizer(text, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.detach().cpu().numpy()[0])
    return dict(zip(topic_labels, probs))

# Inferenz für alle Tweets
print("Calculating topic probabilities...")
topic_scores = musk_twitter_data_nlp['text_lemmatized'].progress_apply(get_topics).apply(pd.Series)

# Neue Spalten anhängen
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), topic_scores], axis=1)

# Ungewichtete Tagesaggregation
print("Aggregating daily topics...")
topics_daily = (
    musk_twitter_data_nlp
    .groupby('date')[topic_labels]
    .mean()
    .reset_index()
)

# Gewichtete Tagesaggregation
print("Aggregating daily topics (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{top}_w": musk_twitter_data_nlp[top] * musk_twitter_data_nlp["engagement_index"]
        for top in topic_labels
    })
    .groupby("date")
    .agg(
        **{f"{top}_w_sum": (f"{top}_w", "sum") for top in topic_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

topics_daily_weighted = (
    weighted_sums
    .assign(**{
        top: weighted_sums[f"{top}_w_sum"] / weighted_sums["total_engagement"]
        for top in topic_labels
    })
    [["date", *topic_labels]]
)

print("Done!")


tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

C:\Users\chrii\PycharmProjects\COIN2025_VollVertnuetelt\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chrii\.cache\huggingface\hub\models--cardiffnlp--tweet-topic-21-multi. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Calculating topic probabilities...


  1%|          | 222/41823 [00:03<09:44, 71.20it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

100%|██████████| 41823/41823 [10:04<00:00, 69.24it/s]


Aggregating daily topics...
Aggregating daily topics (weighted)...
Done!


In [23]:
display(sentiment_daily_weighted.head())
display(emotion_daily_weighted.head())
display(emotion_daily.head())

display(personality_daily_weighted.head())
display(personality_daily.head())
display(daily_word_counts.head())
display(topics_daily_weighted.head())


,date,neg,neu,pos,polarized,nlp_tweet_count
0,2015-01-05,0.022617,0.932019,0.045365,0.000000,2
1,2015-01-06,0.187340,0.765134,0.047527,0.000000,3
2,2015-01-10,0.116972,0.335368,0.547661,0.505779,14
3,2015-01-11,0.006167,0.408818,0.585015,0.000000,1
4,2015-01-12,0.001707,0.288137,0.710156,1.000000,1


,date,anger,disgust,fear,joy,neutral,sadness,surprise
0,2015-01-05,0.011659,0.006370,0.169555,0.009082,0.736129,0.017076,0.050130
1,2015-01-06,0.073970,0.012328,0.292509,0.005659,0.424278,0.020303,0.170953
2,2015-01-10,0.015827,0.022982,0.031871,0.078375,0.640803,0.119906,0.090236
3,2015-01-11,0.058678,0.013213,0.053314,0.026049,0.807787,0.008593,0.032366
4,2015-01-12,0.019790,0.008973,0.107634,0.059775,0.459074,0.295042,0.049712


,date,anger,disgust,fear,joy,neutral,sadness,surprise
0,2015-01-05,0.011031,0.004537,0.110755,0.009524,0.758017,0.021058,0.085079
1,2015-01-06,0.080225,0.012288,0.322267,0.006044,0.411839,0.019574,0.147763
2,2015-01-10,0.011745,0.017963,0.045058,0.096789,0.623210,0.082709,0.122528
3,2015-01-11,0.058678,0.013213,0.053314,0.026049,0.807787,0.008593,0.032366
4,2015-01-12,0.019790,0.008973,0.107634,0.059775,0.459074,0.295042,0.049712


,date,Extroversion,Neuroticism,Agreeableness,Conscientiousness,Openness
0,2015-01-05,0.495897,0.523387,0.427635,0.286978,0.537779
1,2015-01-06,0.478903,0.560536,0.437822,0.286680,0.511013
2,2015-01-10,0.481859,0.530753,0.484134,0.331939,0.501746
3,2015-01-11,0.607662,0.585346,0.384552,0.232823,0.592491
4,2015-01-12,0.515312,0.557470,0.405895,0.266259,0.541958


,date,Extroversion,Neuroticism,Agreeableness,Conscientiousness,Openness
0,2015-01-05,0.512569,0.529826,0.424951,0.289537,0.539911
1,2015-01-06,0.475949,0.560234,0.436405,0.285027,0.510752
2,2015-01-10,0.488609,0.539339,0.474141,0.316610,0.495148
3,2015-01-11,0.607662,0.585346,0.384552,0.232823,0.592491
4,2015-01-12,0.515312,0.557470,0.405895,0.266259,0.541958


tokens,tesla,stock,market,price,profit,loss,revenue,inflation,interest,bitcoin,dogecoin,crypto,ethereum,spacex,model,cybertruck,starship,buy,sell
date,,,,,,,,,,,,,,,,,,,
2015-01-29,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2015-02-24,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2015-03-11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0
2015-03-16,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2015-03-17,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,date,arts_culture,business_entrepreneurs,celebrity_pop_culture,diaries_daily_life,family,fashion_style,film_tv_video,fitness_&_health,food_&_dining,gaming,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life
0,2015-01-05,0.000721,0.002525,0.002338,0.017064,0.000248,0.000193,0.005307,0.000259,0.000317,0.000745,0.001767,0.000717,0.909028,0.004948,0.000479,0.047566,0.001076,0.003952,0.000748
1,2015-01-06,0.002582,0.004803,0.005481,0.042273,0.000923,0.000324,0.006473,0.001162,0.000768,0.001088,0.004947,0.001789,0.568328,0.008736,0.001778,0.337668,0.005458,0.003663,0.001758
2,2015-01-10,0.008144,0.029774,0.007544,0.107567,0.001327,0.001670,0.097501,0.001526,0.002084,0.008087,0.011691,0.015394,0.230218,0.033266,0.002195,0.397744,0.008771,0.032834,0.002665
3,2015-01-11,0.000345,0.021637,0.000819,0.000858,0.000303,0.000242,0.001516,0.001210,0.000663,0.000621,0.005107,0.000580,0.067496,0.001125,0.000365,0.893970,0.000827,0.000938,0.001376
4,2015-01-12,0.014893,0.006263,0.010204,0.018899,0.002197,0.001489,0.012717,0.002197,0.001750,0.004554,0.029326,0.012859,0.104657,0.016331,0.002606,0.713392,0.003677,0.035945,0.006044


## Additional Features to consider/ ToDos

- ToDo Tweet-Typ (z. B. Meme, Information, Ankündigung, Meinung, Engagement)
    - Studien zeigen, dass z. B. Meme-Posts und ironische Tweets besonders starke Kursreaktionen auslösen 
    - Bei Musk besonders relevant, da sein Kommunikationsstil sich im Zeitverlauf stark verändert hat 


Aus Termin mit Peter:
- Einflussreiche weitere Personen: Kann man ggf auch aus quotes nehmen, ist mir nicht mehr ganz klar was er wollte.
- Quotes mit einbeziehen, Quote dataset enthält die texte der Quotes -> Einbeziehen, höhere Genauigkeit bei eg toics

# Create final Daily DF
One can just add Features to the existing Dataframe or create a new Final Daily Df
### Add new Features to existing dataframe

In [24]:
if einzelne_features_zur_bestehenden_CSV_hinzufügen:
    # 1. Final-Dataset laden mit geparster Datumsspalte
    final_daily_df = pd.read_csv("Data/twitter_data/processed/final_daily_df.csv", parse_dates=["date"])

    # 2. Platzhaltervariable für zusätzliche Feature-DataFrames
    # Beispiel: zusatz_feature_dfs = [df_neues_feature_1, df_neues_feature_2, ...]
    zusatz_feature_dfs = [
        
        # HIER DIE OBEN ERSTELLTEN NEUEN SPALTEN (inkl. 'date' spalte) AUFLISTEN
        engagement_metrics
    ]

    # 3. Iterativ mergen
    for feature_df in zusatz_feature_dfs:
        
        feature_df["date"] = pd.to_datetime(feature_df["date"])
        
        # Prüfen auf doppelte Spalten (außer 'date')
        doppelte = [col for col in feature_df.columns if col != "date" and col in final_daily_df.columns]
        if doppelte:
            raise ValueError(f"Die folgenden Spalten sind bereits in final_daily_df vorhanden und sollten evtl. nicht erneut gemerged werden: {doppelte}")

        # Merge auf 'date'
        final_daily_df = pd.merge(final_daily_df, feature_df, on="date", how="left")

    # 4. Ergebnis zurückschreiben
    final_daily_df.to_csv("Data/twitter_data/processed/final_daily_df.csv", index=False)


### Merge and Create final df
Merge the daily dfs in one new dataframe and create csv (creates a weighted and an unweighted version)

In [25]:
# Merge with complete date, fill missing days with zero
if vollstaendige_neuerstellung_der_csv:
    # Unweighted final daily DataFrame
    final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    final_daily_df = final_daily_df.merge(engagement_metrics, on="date", how="left")
    final_daily_df["tweet_count"] = final_daily_df["tweet_count"].astype(int)
    final_daily_df = final_daily_df.merge(sentiment_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(emotion_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(personality_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(daily_word_counts, on="date", how="left")
    final_daily_df = final_daily_df.merge(topics_daily, on="date", how="left")
    final_daily_df["no_tweets"] = (final_daily_df["tweet_count"] == 0).astype(int)
    display(final_daily_df.info())
    display(final_daily_df.head())
    # Weighted final daily DataFrame
    weighted_final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    weighted_final_daily_df = weighted_final_daily_df.merge(engagement_metrics, on="date", how="left")
    weighted_final_daily_df["tweet_count"] = weighted_final_daily_df["tweet_count"].astype(int)
    weighted_final_daily_df = weighted_final_daily_df.merge(sentiment_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(emotion_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(personality_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(daily_word_counts, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(topics_daily_weighted, on="date", how="left")
    weighted_final_daily_df["no_tweets"] = (weighted_final_daily_df["tweet_count"] == 0).astype(int)

    display(weighted_final_daily_df.info())
    display(weighted_final_daily_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3756 entries, 0 to 3755
Data columns (total 62 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    3756 non-null   object 
 1   tweet_count             3756 non-null   int32  
 2   likeCount               3056 non-null   float64
 3   quoteCount              3056 non-null   float64
 4   retweetCount            3056 non-null   float64
 5   replyCount              3056 non-null   float64
 6   neg                     2999 non-null   float32
 7   neu                     2999 non-null   float32
 8   pos                     2999 non-null   float32
 9   nlp_tweet_count         2999 non-null   float64
 10  polarized               2999 non-null   float64
 11  anger                   2999 non-null   float32
 12  disgust                 2999 non-null   float32
 13  fear                    2999 non-null   float32
 14  joy                     2999 non-null   

None

,date,tweet_count,likeCount,quoteCount,retweetCount,replyCount,neg,neu,pos,nlp_tweet_count,...,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life,no_tweets
0,2015-01-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2015-01-02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2015-01-03,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2015-01-04,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2015-01-05,2,3575.0,3.0,3625.0,400.0,0.020146,0.926847,0.053006,2.0,...,0.003858,0.001551,0.796911,0.011698,0.001117,0.104952,0.002432,0.009344,0.001631,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3756 entries, 0 to 3755
Data columns (total 62 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    3756 non-null   object 
 1   tweet_count             3756 non-null   int32  
 2   likeCount               3056 non-null   float64
 3   quoteCount              3056 non-null   float64
 4   retweetCount            3056 non-null   float64
 5   replyCount              3056 non-null   float64
 6   neg                     2999 non-null   float64
 7   neu                     2999 non-null   float64
 8   pos                     2999 non-null   float64
 9   polarized               2999 non-null   float64
 10  nlp_tweet_count         2999 non-null   float64
 11  anger                   2999 non-null   float64
 12  disgust                 2999 non-null   float64
 13  fear                    2999 non-null   float64
 14  joy                     2999 non-null   

None

,date,tweet_count,likeCount,quoteCount,retweetCount,replyCount,neg,neu,pos,polarized,...,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life,no_tweets
0,2015-01-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2015-01-02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2015-01-03,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2015-01-04,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2015-01-05,2,3575.0,3.0,3625.0,400.0,0.022617,0.932019,0.045365,0.0,...,0.001767,0.000717,0.909028,0.004948,0.000479,0.047566,0.001076,0.003952,0.000748,0


### Export

In [26]:
if vollstaendige_neuerstellung_der_csv:
    final_daily_df.to_csv(os.path.join('processed', 'final_daily_df.csv'), index=False)
    weighted_final_daily_df.to_csv(os.path.join('processed', 'weighted_final_daily_df.csv'), index=False)